# 1. 환경 설정 (수정 금지)

In [ ]:
## rdkit 라이브러리 설치
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 38.9 MB/s eta 0:00:00


In [ ]:
## rdkit 설치 확인 및 로그 설정
import rdkit
rdkit.RDLogger.DisableLog('rdApp.*')  # Suppress RDKit warnings

## random seed 고정
import random
import numpy as np
import torch

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 주요 라이브러리 버전 확인
print(torch.__version__)
print(np.__version__)
print(rdkit.__version__)

2.10.0+cpu
2.0.2
2026.03.2


# 2. 문제 정의 (수정 금지)

In [ ]:
## Vocabulart 정의
class Vocab:
    ## Special tokens
    START_TOK = '<START>'
    END_TOK = '<END>'

    ## All tokens
    tokens = [
        START_TOK, # 0. 시퀀스 시작 토큰
        END_TOK,  # 1. 시퀀스 종료 토큰
        'C',      # 2. 탄소 원자
        'N',      # 3. 질소 원자
        'O',      # 4. 산소 원자
        '-',      # 5. 단일 결합
        '=',      # 6. 이중 결합
        '#',      # 7. 삼중 결합
        '(',      # 8. 분기 시작
        ')',      # 9. 분기 종료
        '1',      # 10. 링 번호 1
        '2',      # 11. 링 번호 2
    ]

    ## Token to index, index to token
    token_to_idx = {t:i for i, t in enumerate(tokens)}
    idx_to_token = {i:t for t, i in token_to_idx.items()}

    ## Special indices
    START_IDX = token_to_idx[START_TOK]
    END_IDX = token_to_idx[END_TOK]

    ## Number of tokens
    size = len(tokens)

    @classmethod
    def t2i(cls, x: str) -> int:
        return cls.token_to_idx[x]

    @classmethod
    def i2t(cls, x: int) -> str:
        return cls.idx_to_token[x]

    @classmethod
    def encode(cls, smi: str) -> list[int]:
        processed_smi = smi

        # 시퀀스 시작 토큰이 이미 있으면 제거
        if processed_smi.startswith(cls.START_TOK):
            processed_smi = processed_smi[len(cls.START_TOK):]

        # 시퀀스 종료 토큰이 이미 있으면 제거
        if processed_smi.endswith(cls.END_TOK):
            processed_smi = processed_smi[:-len(cls.END_TOK)]

        # 인덱스 시퀀스 생성
        return [cls.START_IDX] + [cls.t2i(token) for token in processed_smi] + [cls.END_IDX]

    @classmethod
    def decode(cls, indices: list[int], include_special_tokens: bool = False) -> str:
        tokens = []
        for idx in indices:
            token = cls.i2t(idx)
            if token == cls.END_TOK:
                if include_special_tokens:
                    tokens.append(token)
                break # 종료 토큰을 만나면 디코딩 중지
            if token == cls.START_TOK:
                if include_special_tokens:
                    tokens.append(token)
            else:
                tokens.append(token)
        return ''.join(tokens)

######################################################
## Examples
######################################################
print(Vocab.tokens)
print(Vocab.size)
print(Vocab.encode('CC(C)C'))
print(Vocab.encode('<START>CC(C)C<END>'))
print(Vocab.decode([0, 2, 10, 6, 2, 2, 6, 2, 2, 6, 2, 10, 1]))
print(Vocab.decode([0, 2, 10, 6, 2, 2, 6, 2, 2, 6, 2, 10, 1], True))

['<START>', '<END>', 'C', 'N', 'O', '-', '=', '#', '(', ')', '1', '2']
12
[0, 2, 2, 8, 2, 9, 2, 1]
[0, 2, 2, 8, 2, 9, 2, 1]
C1=CC=CC=C1
<START>C1=CC=CC=C1<END>


# 3. 평가 함수 (수정 금지)

In [ ]:
class Evaluator:
    ## 제약 조건
    total_samples = 100
    max_length = 20

    @classmethod
    def is_valid_smiles(cls, smiles: str) -> bool:
        ## 1) Be non-emptu
        if not smiles or len(smiles) == 0:
            return False

        ## 2) Be parseable by RDKit
        try:
            mol = rdkit.Chem.MolFromSmiles(smiles)
            if mol is None:
                return False
        except:
            return False

        ## 3) At least one atom
        if mol.GetNumAtoms() == 0:
            return False

        ## 4) Maximum length
        if len(smiles) > cls.max_length:
            return False

        return True

    @classmethod
    def evaluate(cls, smiles_list: list[str]) -> dict:
        ## total
        n_total = len(smiles_list)
        if n_total != cls.total_samples:
            return {
                'n_total': n_total,
                'n_valid': 0,
                'n_unique': 0,
                'validity': 0.0,
                'uniqueness': 0.0,
                'score': 0.0,
            }

        ## valid
        valid_smiles_list = [smi for smi in smiles_list if cls.is_valid_smiles(smi)]
        n_valid = len(valid_smiles_list)

        ## unique
        unique_smiles_list = list(set(valid_smiles_list))
        n_unique = len(unique_smiles_list)

        return {
            'n_total': n_total,
            'n_valid': n_valid,
            'n_unique': n_unique,
            'validity': n_valid / n_total if n_total > 0 else 0.0,
            'uniqueness': n_unique / n_valid if n_valid > 0 else 0.0,
            'score': n_unique / n_total if n_total > 0 else 0.0,
        }

######################################################
## Examples
######################################################
print(Evaluator.evaluate(['C', 'CC', 'C', 'CCO', 'C=O', 'C#N', '', 'CC(', 'C1CC', '12']))
print(Evaluator.evaluate(['C', 'CC', 'C', 'CCO', 'C=O', 'C#N', '', 'CC(', 'C1CC', '12'] * 10))

{'n_total': 10, 'n_valid': 0, 'n_unique': 0, 'validity': 0.0, 'uniqueness': 0.0, 'score': 0.0}
{'n_total': 100, 'n_valid': 60, 'n_unique': 5, 'validity': 0.6, 'uniqueness': 0.08333333333333333, 'score': 0.05}


# 4. 환경(Env) 클래스 구현

In [ ]:
class Env:
    def __init__(self):
        # action space 크기
        self.action_size = Vocab.size

        # 최대 길이
        self.max_length = Evaluator.max_length

        # 비어있는 칸에 들어갈 값 (<END>)
        self.pad_idx = Vocab.END_IDX

        # 최근 성공한 고유 분자를 기억할 세트
        self.generated_pool = set()

        self.reset()

    def reset(self):
        self.done = False
        # 패딩으로 채운 고정크기 배열 생성
        self.state = np.full(self.max_length, self.pad_idx, dtype=np.int64)

        # 첫 번째 자리에만 토큰 배치
        self.state[0] = Vocab.START_IDX

        # 토큰이 채워진 인덱스 추적값
        self.current_len = 1

        # 텐서로 반환
        return torch.tensor(self.state, dtype=torch.long)

    def step(self, action: int):
        if self.done:
              raise RuntimeError("Episode Ended")

        # 액션 저장
        self.state[self.current_len] = action

        # 토큰 추적 인덱스 업데이트
        self.current_len += 1

        # 종료조건 업데이트
        self.done = (action == Vocab.END_IDX) or (self.current_len >= self.max_length)

        # 다음 상태 반환
        reward = 0.0

        # 생성 기록을 저장할 로그 Dict
        log_info = {}

        if self.done:
            # 패딩 제외한 생성 토큰
            actual_indices = self.state[:self.current_len].tolist()

            # 토큰 디코딩
            smiles = Vocab.decode(actual_indices)

            # 보상 계산
            base_reward = 1.0 if Evaluator.is_valid_smiles(smiles) else -0.1
            is_valid = Evaluator.is_valid_smiles(smiles)

            # 생성 데이터 가이드 보상 계산
            shaping_reward = get_shaping_reward(smiles)

            # 중복 검사
            duplicate_penalty = 0.0
            if is_valid:
                if smiles in self.generated_pool:
                    # 이미 성공했던 분자라면 보상을 대폭 깎아서 다른 탐색을 유도
                    duplicate_penalty = -0.7
                else:
                    # 새로운 유효 분자라면 풀에 추가
                    self.generated_pool.add(smiles)
                    # 풀이 너무 커지면 초기화
                    if len(self.generated_pool) > 500:
                        self.generated_pool.clear()

            # 생성된 SMILE 최종 평가
            reward = base_reward + shaping_reward + duplicate_penalty

            # 로그 저장
            log_info['smiles'] = smiles
            log_info['is_valid'] = Evaluator.is_valid_smiles(smiles)
            log_info['reward'] = reward
            log_info['shaping_reward'] = shaping_reward

        # 상태, 보상, 완료여부, 로그 반환
        return torch.tensor(self.state, dtype=torch.long), reward, self.done, log_info

# 5. 기타 등등

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import pandas as pd
import torch

In [ ]:
#  SMILE 문자열 구조 규칙성 검사
def get_shaping_reward(smiles: str) -> float:
    if not smiles:
        return 0.0

    shaping_reward = 0.0

    #괄호 짝 검사
    paren_count = 0
    paren_valid = True
    for char in smiles:
        if char == '(':
            paren_count += 1
        elif char == ')':
            paren_count -= 1
            if paren_count < 0: # 닫는 괄호가 먼저 나온 경우
                paren_valid = False

    if paren_valid and paren_count == 0:
        shaping_reward += 0.2  # 괄호 짝이 완벽히 맞으면 보너스
    else:
        shaping_reward -= 0.1  # 괄호가 열린 채 끝나거나 꼬이면 페널티

    # 고리 번호 짝 검사
    count_1 = smiles.count('1')
    count_2 = smiles.count('2')

    if count_1 > 0 and count_1 % 2 == 0:
        shaping_reward += 0.1  # 1번 고리가 잘 닫힘
    elif count_1 > 0:
        shaping_reward -= 0.1

    if count_2 > 0 and count_2 % 2 == 0:
        shaping_reward += 0.1  # 2번 고리가 잘 닫힘
    elif count_2 > 0:
        shaping_reward -= 0.1

    # 하나 이상의 원소 포함 여부 검사
    has_atom = any(atom in smiles for atom in ['C', 'N', 'O'])
    if has_atom:
        shaping_reward += 0  # 원소를 정상적으로 사용하면 보너스
    else:
        shaping_reward -= 0.3  # 결합 기호나 괄호만 쓴 경우 패널티

    return shaping_reward

In [ ]:
# 신경망 구성

class A2CNet(nn.Module):
    def __init__(self, vocab_size = 12, embedding_dim = 64, hidden_dim = 128):
        super(A2CNet, self).__init__()

        self.vocab_size = vocab_size

        # 임베딩 레이어
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=Vocab.END_IDX
        )

        # GRU 레이어
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2, # 2층 레이어
            batch_first=True,
            dropout=0.1   # 드롭아웃 0.1 ~
        )

        # 토큰 선택 로짓 출력
        self.actor = nn.Linear(hidden_dim, vocab_size)

        # 가치 출력
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, state, current_length):
        # 임베딩 레이어 통과
        x = self.embedding(state)
        # GRU 통과
        output, _ = self.gru(x)

        # 패딩 전 토큰의 위치 가져오기
        last_step_indices = (current_length - 1).long()

        # 실제 토큰 가져오기
        last_hidden = output[torch.arange(state.size(0)), last_step_indices, :]

        # 로짓, 가치 계산
        logits = self.actor(last_hidden)
        value = self.critic(last_hidden)

        return logits, value

In [ ]:
# 에이전트 구현

class A2CAgent:
    def __init__(self, model, lr=1e-3, gamma=0.99, entropy_coef=0.01, critic_coef=0.5, epsilon = 0.05):
        self.model = model
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.critic_coef = critic_coef
        self.epsilon = epsilon

        # 옵티마이저 설정
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

    def get_action(self, state, current_len):
        # 추론 모드
        self.model.eval()
        with torch.no_grad():
            state_input = state.unsqueeze(0) # 배치 차원 추가: (1, max_length)
            lengths_input = torch.tensor([current_len], dtype=torch.long)
            logits, value = self.model(state_input, lengths_input)

            if np.random.rand() < self.epsilon:
                action = np.random.randint(0, self.model.vocab_size)
            else:
                # 확률 분포 기반 행동 샘플링 (탐색 유도)
                prob = torch.softmax(logits, dim=-1)
                dist = Categorical(prob)
                action = dist.sample().item()

        return action, value.item()

    def evaluate_states(self, states, lengths, actions):
        # 학습 모드
        self.model.train()
        logits, values = self.model(states, lengths)

        prob = torch.softmax(logits, dim=-1)
        dist = Categorical(prob)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        return log_probs, values.squeeze(-1), entropy

    def update(self, loss):
        # 가중치 업데이트
        self.optimizer.zero_grad()
        loss.backward()

        # 그라디언트 클리핑 (학습 안정성 확보)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)

        self.optimizer.step()

In [ ]:
class A2CTrain:
    def __init__(self, env, agent, max_episodes=1000):
        self.env = env
        self.agent = agent
        self.max_episodes = max_episodes

    def train(self):
        print("A2C 분자 생성 학습을 시작합니다.")
        print("-" * 60)

        # 성공률 추적 배열
        success_history = []
        # 다양성 추적용 기록 배열
        smiles_history = []

        for episode in range(1, self.max_episodes + 1):
            states_list = []
            actions_list = []
            rewards_list = []
            lengths_list = []

            # 환경 초기화
            state = self.env.reset()
            done = False

            while not done:
                # 현재까지 채워진 실제 토큰 길이 계산
                current_len = self.env.current_len

                # 에이전트로부터 행동 샘플링
                action, _ = self.agent.get_action(state, current_len)

                # 데이터 저장
                states_list.append(state.clone())
                actions_list.append(action)
                lengths_list.append(current_len)

                # 환경 한 스텝 진행
                next_state, reward, done, info = self.env.step(action)

                rewards_list.append(reward)
                state = next_state

            if done:
                # 로그로 생성된 데이터 가져오기
                is_valid = info.get('is_valid', False)
                generated_smiles = info.get('smiles', 'None')

                success_history.append(1.0 if is_valid else 0.0)
                smiles_history.append(generated_smiles)

            # 누적 이득 계산
            returns_list = []
            G = 0
            for r in reversed(rewards_list):
                G = r + self.agent.gamma * G
                returns_list.insert(0, G)

            # 리스트 데이터를 텐서 배치로 변환
            states = torch.stack(states_list)        # (episode_steps, max_length)
            actions = torch.tensor(actions_list)     # (episode_steps,)
            lengths = torch.tensor(lengths_list)     # (episode_steps,)
            returns = torch.tensor(returns_list, dtype=torch.float32)   # (episode_steps,)

            # 손실 함수 계산 및 에이전트 업데이트
            log_probs, values, entropy = self.agent.evaluate_states(states, lengths, actions)

            # Advantage 계산
            advantages = returns - values.detach()

            # Loss 구성
            actor_loss = -(log_probs * advantages).mean()
            critic_loss = torch.nn.functional.mse_loss(values, returns)
            entropy_loss = entropy.mean()

            # 총 손실 함수
            total_loss = (actor_loss + self.agent.critic_coef * critic_loss - self.agent.entropy_coef * entropy_loss)

            # 에이전트 가중치 업데이트
            self.agent.update(total_loss)

            # 주기적인 모니터링 로그 출력 (100 에피소드마다)
            if episode % 100 == 0:
                # 최근 100회의 성공률 계산
                recent_success = success_history[-100:]
                recent_success_rate = np.mean(recent_success) * 100

                # 최근 100회의 생성 결과물 슬라이싱
                recent_smiles = smiles_history[-100:]

                # 100회의 SMILE 문자열중 중복 없는 분자식만 필터링
                valid_smiles_only = [
                    smi for smi, success in zip(recent_smiles, recent_success) if success == 1
                ]

                # 세트 자료형으로 변환하여 중복 제거 후 개수 측정
                unique_valid_count = len(set(valid_smiles_only))

                # 에피소드에서 최종적으로 얻은 리워드 가져오기
                final_step_reward = rewards_list[-1]

                print(f"Ep {episode:4d} | Loss: {total_loss.item():.4f} | " f"최근100회 성공률: {recent_success_rate:.1f}% | " f"고유 유효분자수: {unique_valid_count}/100 | " f"SMILES: {generated_smiles} | " f"Reward: {final_step_reward:+.1f}")

In [ ]:
# 하이퍼파라미터 정의
VOCAB_SIZE = Vocab.size # 12
EMBEDDING_DIM = 64
HIDDEN_DIM = 128

# 환경, 모델, 에이전트, 트레이너 객체 생성
env = Env()
model = A2CNet(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM)
agent = A2CAgent(model=model, lr=0.001, gamma=0.95, entropy_coef=0.02, epsilon=0.05)
trainer = A2CTrain(env=env, agent=agent, max_episodes=5000)

# 학습 시작
trainer.train()

A2C 분자 생성 학습을 시작합니다.
------------------------------------------------------------
Ep  100 | Loss: -0.0616 | 최근100회 성공률: 2.0% | 고유 유효분자수: 2/100 | SMILES:  | Reward: -0.1
Ep  200 | Loss: 0.3841 | 최근100회 성공률: 5.0% | 고유 유효분자수: 4/100 | SMILES: =O- | Reward: +0.1
Ep  300 | Loss: -0.1427 | 최근100회 성공률: 66.0% | 고유 유효분자수: 44/100 | SMILES: NNOO | Reward: +0.5
Ep  400 | Loss: -0.0272 | 최근100회 성공률: 73.0% | 고유 유효분자수: 56/100 | SMILES: NON | Reward: +0.5
Ep  500 | Loss: 0.0215 | 최근100회 성공률: 83.0% | 고유 유효분자수: 76/100 | SMILES: COOONN | Reward: +1.2
Ep  600 | Loss: -2.3672 | 최근100회 성공률: 81.0% | 고유 유효분자수: 74/100 | SMILES: OCO2CN | Reward: +0.0
Ep  700 | Loss: 0.0527 | 최근100회 성공률: 85.0% | 고유 유효분자수: 76/100 | SMILES: CNCOONO | Reward: +1.2
Ep  800 | Loss: 0.0386 | 최근100회 성공률: 86.0% | 고유 유효분자수: 82/100 | SMILES: OONCO | Reward: +1.2
Ep  900 | Loss: -0.0806 | 최근100회 성공률: 85.0% | 고유 유효분자수: 82/100 | SMILES: NNNCCCCCC | Reward: +1.2
Ep 1000 | Loss: -0.0180 | 최근100회 성공률: 94.0% | 고유 유효분자수: 44/100 | SMILES: NO | Rewa

In [ ]:
def generate_and_save_to_csv(model, env, filename="generated_smiles.csv", num_samples=100):
    print(f"모델 기반 SMILES 문자열 {num_samples}개 생성 시작")
    print("-" * 60)

    generated_list = []

    model.eval()

    for i in range(1, num_samples + 1):
        state = env.reset()
        done = False

        while not done:
            current_len = env.current_len

            with torch.no_grad():
                state_input = state.unsqueeze(0)
                lengths_input = torch.tensor([current_len], dtype=torch.long)
                logits, _ = model(state_input, lengths_input)

                # 확률 분포 기반 샘플링
                prob = torch.softmax(logits, dim=-1)
                dist = torch.distributions.Categorical(prob)
                action = dist.sample().item()

            # 환경 진행
            next_state, reward, done, log_info = env.step(action)
            state = next_state

        # 에피소드가 끝났을 디코딩해둔 최종 SMILES 문자열 추출
        if 'smiles' in log_info:
            smiles_str = log_info['smiles']
            is_valid = log_info['is_valid']

            # 결과 리스트에 저장
            generated_list.append({
                'id': i,
                'smiles': smiles_str,
                'is_valid': is_valid
            })

            if i % 20 == 0:
                print(f">> {i}/{num_samples} 생성 완료...")

    # 판다스 데이터프레임으로 변환
    df = pd.DataFrame(generated_list)

    # CSV 파일로 저장
    df.to_csv(filename, index=False, encoding='utf-8')

    # 결과 점수 출력
    smiles_list = df['smiles'].tolist()
    results = Evaluator.evaluate(smiles_list)

    print("-" * 60)
    print(f"저장 완료, 파일명: {filename}")

    print("\n과제 평가 결과 =========")
    print(f"총 생성 개수   : {results['n_total']}개")
    print(f"유효한 분자수  : {results['n_valid']}개 (유효성: {results['validity']*100:.1f}%)")
    print(f"고유한 분자수  : {results['n_unique']}개 (고유성: {results['uniqueness']*100:.1f}%)")
    print(f"과제 점수 (Score)  : {results['score']:.4f}")
    print("==============================================\n")

    # 주피터 노트북 화면에서 상위 5개 미리보기 출력
    return df.head()

In [ ]:
preview_df = generate_and_save_to_csv(model, env, filename="generated0.csv", num_samples=100)
preview_df

In [ ]:
def evaluate_csv(filename):
    df = pd.read_csv(filename)
    smiles_list = df['smiles'].tolist()
    results = Evaluator.evaluate(smiles_list)

    # 4. 평가 결과 출력하기
    print("📊 [A2C 모델 생성 분자 최종 평가 결과]")
    print("-" * 50)
    print(f"🔹 전체 샘플 수 (Total Samples)  : {results['n_total']}개")
    print(f"🔹 유효한 분자 수 (Valid)        : {results['n_valid']}개 (유효성): {results['validity']*100:.1f}%)")
    print(f"🔹 고유한 분자 수 (Unique)       : {results['n_unique']}개 (고유성: {results['uniqueness']*100:.1f}%)")
    print("-" * 50)
    print(f"🏆 최종 점수 (Score)             : {results['score']:.4f}")
    print("-" * 50)

In [ ]:
evaluate_csv('generated1.csv')

📊 [A2C 모델 생성 분자 최종 평가 결과]
--------------------------------------------------
🔹 전체 샘플 수 (Total Samples)  : 100개
🔹 유효한 분자 수 (Valid)        : 100개 (유효성): 100.0%)
🔹 고유한 분자 수 (Unique)       : 99개 (고유성: 99.0%)
--------------------------------------------------
🏆 최종 점수 (Score)             : 0.9900
--------------------------------------------------
